In [8]:
import scipy.io
import numpy as np

mat = scipy.io.loadmat(
    "../data/raw/delta/tests/data/movie_mothermachine_tif/expected_results/Position000001.mat",
    simplify_cells=False
)

res = mat['res']
print("res shape:", res.shape)

chamber_data = res[0, 0]
print("chamber_data dtype:", chamber_data.dtype)

lineage = chamber_data['lineage'][0, 0]
print("lineage shape:", lineage.shape)
print("lineage dtype:", lineage.dtype)

# Look at first cell
cell = lineage[0, 0]
print("\ncell dtype:", cell.dtype)

# Inspect each field's shape
for field in cell.dtype.names:
    val = cell[field]
    print(f"  {field}: type={type(val)}, shape={getattr(val,'shape','N/A')}, dtype={getattr(val,'dtype','N/A')}")
    # Go one level deeper
    if hasattr(val, 'shape') and val.size > 0:
        inner = val.flat[0]
        print(f"    └─ inner: type={type(inner)}, shape={getattr(inner,'shape','N/A')}")
        if hasattr(inner, 'shape') and inner.size > 0:
            inner2 = inner.flat[0]
            print(f"       └─ inner2: type={type(inner2)}, shape={getattr(inner2,'shape','N/A')}")

res shape: (1, 18)
chamber_data dtype: [('labelsstack_resized', 'O'), ('labelsstack', 'O'), ('lineage', 'O')]
lineage shape: (1, 9)
lineage dtype: object

cell dtype: [('mother', 'O'), ('frames', 'O'), ('daughters', 'O'), ('edges', 'O'), ('area', 'O'), ('width', 'O'), ('length', 'O'), ('perimeter', 'O'), ('old_pole', 'O'), ('new_pole', 'O'), ('growthrate_length', 'O'), ('growthrate_area', 'O')]
  mother: type=<class 'numpy.ndarray'>, shape=(1, 1), dtype=object
    └─ inner: type=<class 'numpy.ndarray'>, shape=(1, 1)
       └─ inner2: type=<class 'numpy.int64'>, shape=()
  frames: type=<class 'numpy.ndarray'>, shape=(1, 1), dtype=object
    └─ inner: type=<class 'numpy.ndarray'>, shape=(1, 10)
       └─ inner2: type=<class 'numpy.float32'>, shape=()
  daughters: type=<class 'numpy.ndarray'>, shape=(1, 1), dtype=object
    └─ inner: type=<class 'numpy.ndarray'>, shape=(1, 10)
       └─ inner2: type=<class 'numpy.float32'>, shape=()
  edges: type=<class 'numpy.ndarray'>, shape=(1, 1), dty

In [9]:
import scipy.io
import numpy as np
import pandas as pd
import os

def parse_delta_mat(mat_path):
    mat = scipy.io.loadmat(mat_path, simplify_cells=False)
    res = mat['res']
    rows = []
    n_chambers = res.shape[1]

    for chamber_idx in range(n_chambers):
        chamber_data = res[0, chamber_idx]
        lineage = chamber_data['lineage'][0, 0]

        if lineage.size == 0:
            continue

        n_cells = lineage.shape[1] if lineage.ndim > 1 else 1

        for cell_idx in range(n_cells):
            cell = lineage[0, cell_idx] if lineage.ndim > 1 else lineage[0]

            try:
                # Each field is wrapped in (1,1) object array → unwrap with [0,0]
                mother_id    = int(cell['mother'][0, 0][0, 0])
                frames       = cell['frames'][0, 0][0].astype(int)
                daughters    = cell['daughters'][0, 0][0]
                edges        = cell['edges'][0, 0]          # may be empty array
                area         = cell['area'][0, 0][0]
                width        = cell['width'][0, 0][0]
                length       = cell['length'][0, 0][0]
                perimeter    = cell['perimeter'][0, 0][0]
                growthrate_l = cell['growthrate_length'][0, 0][0]
                growthrate_a = cell['growthrate_area'][0, 0][0]
                old_pole     = cell['old_pole'][0, 0]       # shape (N, 2)
                new_pole     = cell['new_pole'][0, 0]       # shape (N, 2)

                divided     = int(np.any(daughters > 0))
                was_ejected = 1 if (
                    edges.size > 0 and (
                        np.any(edges == '+y') or np.any(edges == ' +y')
                    )
                ) else 0

                for t_idx, frame in enumerate(frames):
                    rows.append({
                        'mat_file'    : os.path.basename(mat_path),
                        'chamber_idx' : chamber_idx,
                        'cell_id'     : cell_idx,
                        'mother_id'   : mother_id,
                        'frame'       : int(frame),
                        'area'        : float(area[t_idx]),
                        'width'       : float(width[t_idx]),
                        'length'      : float(length[t_idx]),
                        'perimeter'   : float(perimeter[t_idx]),
                        'growthrate_l': float(growthrate_l[t_idx]),
                        'growthrate_a': float(growthrate_a[t_idx]),
                        'old_pole_y'  : float(old_pole[t_idx, 0]),
                        'old_pole_x'  : float(old_pole[t_idx, 1]),
                        'new_pole_y'  : float(new_pole[t_idx, 0]),
                        'new_pole_x'  : float(new_pole[t_idx, 1]),
                        'divided'     : divided,
                        'was_ejected' : was_ejected,
                    })

            except Exception as e:
                print(f"  Skipping cell {cell_idx} in chamber {chamber_idx}: {e}")

    return pd.DataFrame(rows)


# Run on all MAT files
DELTA_PATH = "../data/raw/delta"
all_dfs = []

for root, dirs, files in os.walk(DELTA_PATH):
    for f in files:
        if f.endswith('.mat'):
            path = os.path.join(root, f)
            print(f"Parsing: {path}")
            try:
                df = parse_delta_mat(path)
                if len(df) > 0:
                    print(f"  → {len(df)} rows, {df['cell_id'].nunique()} unique cell_ids")
                    all_dfs.append(df)
                else:
                    print(f"  → 0 rows (empty)")
            except Exception as e:
                print(f"  → Failed: {e}")

combined_df = pd.concat(all_dfs, ignore_index=True)
print("\nCombined DataFrame shape:", combined_df.shape)
print("\nFirst 5 rows:")
print(combined_df.head(5).to_string())
print("\nColumns:", list(combined_df.columns))
print("\nDivision label distribution:")
print(combined_df.groupby('mat_file')['divided'].value_counts())
print("\nTotal unique cells:", combined_df.groupby(['mat_file','chamber_idx','cell_id']).ngroups)

Parsing: ../data/raw/delta\tests\data\movie_2D_nd2\test_expected_results\Position000000.mat
  → 14 rows, 2 unique cell_ids
Parsing: ../data/raw/delta\tests\data\movie_2D_tif\expected_results\Position000001.mat
  → 4 rows, 1 unique cell_ids
Parsing: ../data/raw/delta\tests\data\movie_mothermachine_tif\expected_results\Position000001.mat
  → 929 rows, 12 unique cell_ids
Parsing: ../data/raw/delta\tests\data\movie_mothermachine_tif\expected_results\Position000002.mat
  → 924 rows, 13 unique cell_ids

Combined DataFrame shape: (1871, 17)

First 5 rows:
             mat_file  chamber_idx  cell_id  mother_id  frame   area    width     length  perimeter  growthrate_l  growthrate_a  old_pole_y  old_pole_x  new_pole_y  new_pole_x  divided  was_ejected
0  Position000000.mat            0        0          0      1  157.0  8.00000  26.000000  60.970562      0.109199      0.067718       261.0       366.0       262.0       344.0        1            0
1  Position000000.mat            0        0      

In [10]:
print(combined_df['was_ejected'].value_counts())

was_ejected
0    1707
1     164
Name: count, dtype: int64


In [11]:
import skimage
print(skimage.__version__)

0.26.0


In [12]:
# Cell 1 — check what's actually in the file
with open('../src/preprocessing.py', 'r') as f:
    print(f.read())

import numpy as np
import cv2
from skimage import restoration

def normalize_image(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    mean = img.mean()
    std = img.std()
    if std < 1e-8:
        return np.zeros_like(img)
    return (img - mean) / std


def subtract_background(img: np.ndarray, radius: int = 50) -> np.ndarray:
    background = restoration.rolling_ball(img, radius=radius)
    return img - background


def preprocess_image(img: np.ndarray, radius: int = 50) -> np.ndarray:
    img_norm = normalize_image(img)
    img_clean = subtract_background(img_norm, radius=radius)
    return img_clean


In [13]:
def subtract_background(img: np.ndarray, radius: int = 50) -> np.ndarray:
    """
    Rolling-ball background subtraction.
    For phase-contrast (dark cells on gray background), 
    we need to invert, subtract, then invert back.
    """
    from skimage import restoration
    
    # Shift to positive range before rolling ball
    img_shifted = img - img.min()
    background = restoration.rolling_ball(img_shifted, radius=radius)
    corrected = img_shifted - background
    
    # Re-center to zero mean
    corrected = corrected - corrected.mean()
    return corrected

In [14]:
# Cell 3 — reload module and test
import importlib
import sys

# Remove cached version if already imported
if 'preprocessing' in sys.modules:
    del sys.modules['preprocessing']

from preprocessing import preprocess_image

import tifffile
import matplotlib.pyplot as plt

img_raw = tifffile.imread('../data/raw/delta/tests/data/movie_2D_tif/Position01Channel01Frame000001.tif')
img_clean = preprocess_image(img_raw)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(img_raw, cmap='gray')
axes[0].set_title(f'Raw — min={img_raw.min()}, max={img_raw.max()}')
axes[1].imshow(img_clean, cmap='gray')
axes[1].set_title(f'Preprocessed — mean={img_clean.mean():.4f}, std={img_clean.std():.4f}')
plt.tight_layout()
plt.savefig('../outputs/figures/week2_preprocessing_example.png', dpi=150)
plt.show()

print(f"Raw dtype: {img_raw.dtype}, shape: {img_raw.shape}")
print(f"Preprocessed mean: {img_clean.mean():.4f}")
print(f"Preprocessed std: {img_clean.std():.4f}")
print("Preprocessing test passed.")

ModuleNotFoundError: No module named 'preprocessing'